# Sphere Octant Division (`N^2`)

Divide the first sphere octant (`x, y, z >= 0`) into `N²` spherical triangles and visualize them.

- The mesh is built from a simplex lattice and normalized onto the sphere.
- Visualization uses `matplotlib` 3D plotting.
- The last cell checks side-length patterns.


## `lattice_to_octant_point`

For lattice index $(i,j,k)$ with $i+j+k=N$, define:

$$\begin{aligned}
r_{ij}&=i+j,  r_{jk}=j+k,  r_{ki}=k+i\\
\theta_{ab}&=\frac{\pi \cdot r_{ab}}{2 N}\quad (ab\in\{ij,jk,ki\})\\
\phi_{ab}&=\begin{cases}0&(r_{ab}=0)\\ \frac{\pi \cdot b}{2 r_{ab}}&(r_{ab}>0)\end{cases}
\end{aligned}$$

Then the output point is:

$$\mathbf{p}(i,j,k)=\frac{1}{3}\begin{bmatrix}
\sin\theta_{ij}\cos\phi_{ij}+\sin\theta_{ki}\sin\phi_{ki}+\cos\theta_{jk}\\
\sin\theta_{jk}\cos\phi_{jk}+\sin\theta_{ij}\sin\phi_{ij}+\cos\theta_{ki}\\
\sin\theta_{ki}\cos\phi_{ki}+\sin\theta_{jk}\sin\phi_{jk}+\cos\theta_{ij}
\end{bmatrix}. $$


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

import numpy as np

from sphere_division_algorithms import (
    build_octant_mesh,
    build_point_index,
    lattice_permutation_error,
    outward_normals_check,
    planar_triangle_areas,
)
from sphere_division_visualization import (
    plot_octant_division, 
    plot_planar_area_distribution
)


## visualization

In [ ]:
# Shared configuration
N = 16
points, triangle_keys, tris = build_octant_mesh(N)
point_keys, point_index = build_point_index(points)
output_path = PROJECT_ROOT / 'figures' / 'octant_n_squared_division.svg'


In [ ]:
# Example run
plot_octant_division(n=N, save_path=output_path)
print(f'number of spherical triangles = {len(tris)} (expected: {N**2})')


In [ ]:
# List all coordinate points (with indices) and all triangle vertex indices
print('=== Points (index: lattice_index -> [x, y, z]) ===')
for key in point_keys:
    idx = point_index[key]
    i, j = key
    x, y, z = points[i, j]
    print(f'{idx:3d}: {key} -> [{x:.8f}, {y:.8f}, {z:.8f}]')

print('\n=== Triangles (triangle_id: point_index_triplet) ===')
for t_id, tri in enumerate(triangle_keys):
    ids = tuple(point_index[v] for v in tri)
    print(f'{t_id:3d}: {ids}')

print(f'\npoint_count = {len(point_keys)}')
print(f'triangle_count = {len(triangle_keys)} (expected: {N**2})')

all_outward, inward_ids = outward_normals_check(points, triangle_keys)
print(f'\nall_outward_normals = {all_outward}')
if not all_outward:
    print('inward_triangle_ids =', inward_ids)

max_perm_err, worst_case = lattice_permutation_error(N)
print(f'max_permutation_error = {max_perm_err:.3e}')
print('worst_case =', worst_case)

In [ ]:
# Planar triangle area distribution
areas = planar_triangle_areas(points, triangle_keys)

print('=== Planar Triangle Area Distribution ===')
print(f'triangle_count = {areas.size}')
print(f'min   = {areas.min():.10f}')
print(f'max   = {areas.max():.10f}')
print(f'mean  = {areas.mean():.10f}')
print(f'median= {np.median(areas):.10f}')
print(f'std   = {areas.std(ddof=0):.10f}')

q = np.quantile(areas, [0.0, 0.25, 0.5, 0.75, 1.0])
print('quantiles (0,25,50,75,100)% =', [f'{v:.10f}' for v in q])

plot_planar_area_distribution(areas, n=N)
